# Фреймворк для оценки агентов: Сравнение

Сравнение трех запусков системы:
1. **ReAct v1 (Баги)**: Агент со сломанным поиском рейсов.
2. **ReAct v2 (Исправлено)**: Агент с работающим поиском.
3. **Multi-Agent System (MAS)**: Иерархический оркестратор с Критиком.

Запуск на 27 задачах с проверкой на эталонных ответах:
- Точность (% совпадения)
- Cohen's Kappa (κ)
- Время (с)
- Затраты ($)

## 1. Настройка окружения

In [1]:
import os
import sys
import time
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv(dotenv_path='../.env')

from src.config import get_llm, REACT_SYSTEM_PROMPT
from src.tools import ALL_TOOLS
from src.graph import build_react_graph, get_multi_agent_system
from src.evaluation import (
    TASKS,
    HUMAN_GT,
    initialize_scoreboard,
    update_scoreboard,
    print_scoreboard,
    run_with_iron_user,
    call_judge_v4,
    cohens_kappa,
    quality_score,
    kappa_interpretation
)

llm = get_llm(temperature=0.1)
print(f"Оценки загружены. Всего задач: {len(TASKS)}")

avg_human_score = sum(quality_score(HUMAN_GT[t["id"]]) for t in TASKS) / len(TASKS)
initialize_scoreboard(avg_human_score)
print(f"Базовое качество эталонных ответов: {avg_human_score:.2f}/6")

c:\Users\alexander\.conda\envs\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\alexander\.conda\envs\llm_env\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Оценки загружены. Всего задач: 27
Базовое качество эталонных ответов: 5.07/6


## 2. Запуск ReAct v1 (Сломанные инструменты)

In [5]:
os.environ["BUGGY_TOOLS"] = "true"

react_v1_graph = build_react_graph(llm, REACT_SYSTEM_PROMPT, ALL_TOOLS)

results_v1 = {}
t0 = time.time()
n_calls = 0

print("=== Запуск ReAct v1 ===")
for idx, task in enumerate(TASKS, 1):
    trajectory = run_with_iron_user(task, react_v1_graph, None, max_turns=5, max_steps=20, llm=llm)
    results_v1[task["id"]] = trajectory
    n_calls += len([s for s in trajectory.steps if s.type == "action"])
    print(f"  Прогресс: {idx}/{len(TASKS)} - Задача {task['id']} завершена")

time_v1 = time.time() - t0
cost_v1 = n_calls * 0.00025
print(f"\nReAct v1 завершил работу за {time_v1:.1f}с, вызовов инструментов: {n_calls}")

=== Запуск ReAct v1 ===
[LOG] user: Show information about booking ABC123
[GUARD] is_on_topic -> yes
  Прогресс: 1/27 - Задача t01 завершена
[LOG] user: Find flights from Moscow to Paris on February 20
[GUARD] is_on_topic -> yes
  Прогресс: 2/27 - Задача t02 завершена
[LOG] user: What is the status of booking XYZ789?
[GUARD] is_on_topic -> yes
  Прогресс: 3/27 - Задача t03 завершена
[LOG] user: I want to move booking ABC123 to a later flight on the same day
[GUARD] is_on_topic -> yes
[LOG] user: I'd like to change my booking to SU2458 departing at 18:30.
  Прогресс: 4/27 - Задача t04 завершена
[LOG] user: Cancel my booking ABC123
[GUARD] is_on_topic -> yes
[LOG] user: Yes, I want to cancel booking ABC123
  Прогресс: 5/27 - Задача t05 завершена
[LOG] user: Find flights to London and compare with Paris on price on February 20
[GUARD] is_on_topic -> yes
  Прогресс: 6/27 - Задача t06 завершена
[LOG] user: Move booking ABC123 to the afternoon flight at 12:30
[GUARD] is_on_topic -> yes
  Про

### Оценка ReAct v1 (Судья v4)

In [6]:
judge_labels_v1 = {}
print("=== Оценка ReAct v1 ===")
for task in TASKS:
    tid = task["id"]
    judge_labels_v1[tid] = call_judge_v4(task["query"], results_v1[tid], llm=llm)

avg_score_v1 = sum(quality_score(judge_labels_v1[t["id"]]) for t in TASKS) / len(TASKS)

human_use = [HUMAN_GT[t["id"]]["usefulness"] for t in TASKS]
judge_use_v1 = [judge_labels_v1[t["id"]]["usefulness"] for t in TASKS]
kappa_use = cohens_kappa(human_use, judge_use_v1)

update_scoreboard("ReAct v1 (Баги)", avg_score_v1, kappa_use, time_v1, n_calls, cost_v1)
print_scoreboard()

=== Оценка ReAct v1 ===

                             EVALUATION SCOREBOARD                              
Version                   Avg Score    Cohen Kappa  Time (s)   Calls    Cost ($)
────────────────────────────────────────────────────────────────────────────────
Human GT                  5.07         1.000        0.0s       0        $0.00000
ReAct v1 (Баги)           5.26         0.108        150.7s     48       $0.01200



## 3. Запуск ReAct v2 (Исправленные инструменты)

In [7]:
os.environ["BUGGY_TOOLS"] = "false"

react_v2_graph = build_react_graph(llm, REACT_SYSTEM_PROMPT, ALL_TOOLS)

results_v2 = {}
t0 = time.time()
n_calls = 0

print("=== Запуск ReAct v2 ===")
for idx, task in enumerate(TASKS, 1):
    trajectory = run_with_iron_user(task, react_v2_graph, None, max_turns=5, max_steps=30, llm=llm)
    results_v2[task["id"]] = trajectory
    n_calls += len([s for s in trajectory.steps if s.type == "action"])
    print(f"  Прогресс: {idx}/{len(TASKS)} - Задача {task['id']} завершена")

time_v2 = time.time() - t0
cost_v2 = n_calls * 0.00025
print(f"\nReAct v2 завершил работу за {time_v2:.1f}с, вызовов инструментов: {n_calls}")

=== Запуск ReAct v2 ===
[LOG] user: Show information about booking ABC123
[GUARD] is_on_topic -> yes
  Прогресс: 1/27 - Задача t01 завершена
[LOG] user: Find flights from Moscow to Paris on February 20
[GUARD] is_on_topic -> yes
  Прогресс: 2/27 - Задача t02 завершена
[LOG] user: What is the status of booking XYZ789?
[GUARD] is_on_topic -> yes
  Прогресс: 3/27 - Задача t03 завершена
[LOG] user: I want to move booking ABC123 to a later flight on the same day
[GUARD] is_on_topic -> yes
[LOG] user: I'd like to choose the second option, SU2458 at 18:30.
  Прогресс: 4/27 - Задача t04 завершена
[LOG] user: Cancel my booking ABC123
[GUARD] is_on_topic -> yes
[LOG] user: Yes, I want to cancel booking ABC123
  Прогресс: 5/27 - Задача t05 завершена
[LOG] user: Find flights to London and compare with Paris on price on February 20
[GUARD] is_on_topic -> yes
  Прогресс: 6/27 - Задача t06 завершена
[LOG] user: Move booking ABC123 to the afternoon flight at 12:30
[GUARD] is_on_topic -> yes
[LOG] user

### Оценка ReAct v2 (Судья v4)

In [8]:
judge_labels_v2 = {}
print("=== Оценка ReAct v2 ===")
for task in TASKS:
    tid = task["id"]
    judge_labels_v2[tid] = call_judge_v4(task["query"], results_v2[tid], llm=llm)

avg_score_v2 = sum(quality_score(judge_labels_v2[t["id"]]) for t in TASKS) / len(TASKS)
judge_use_v2 = [judge_labels_v2[t["id"]]["usefulness"] for t in TASKS]
kappa_use_v2 = cohens_kappa(human_use, judge_use_v2)

update_scoreboard("ReAct v2 (Исправлено)", avg_score_v2, kappa_use_v2, time_v2, n_calls, cost_v2)
print_scoreboard()

=== Оценка ReAct v2 ===

                             EVALUATION SCOREBOARD                              
Version                   Avg Score    Cohen Kappa  Time (s)   Calls    Cost ($)
────────────────────────────────────────────────────────────────────────────────
Human GT                  5.07         1.000        0.0s       0        $0.00000
ReAct v1 (Баги)           5.26         0.108        150.7s     48       $0.01200
ReAct v2 (Исправлено)     4.85         0.111        163.9s     57       $0.01425



## 4. Запуск Multi-Agent System (MAS)

In [9]:
from src.database import DB, fresh_db, snapshot_db
from src.evaluation import Step, Trajectory

mas_coordinator = get_multi_agent_system(llm, max_revisions=1)

results_mas = {}
t0 = time.time()
n_calls = 0

print("=== Запуск Multi-Agent System (MAS) ===")
for idx, task in enumerate(TASKS, 1):
    DB.clear()
    DB.update(fresh_db())
    db_before = snapshot_db(DB)
    
    t_start = time.time()
    final_answer, specialist_results, critic_feedback = mas_coordinator.process_query_with_qc(task["query"])
    
    t_obj = Trajectory(task_id=task["id"], query=task["query"], db_before=db_before, final_response=final_answer)
    
    for res in specialist_results:
        for tool_name in res.tools_used:
            t_obj.steps.append(Step("action", {"tool": tool_name}))
            n_calls += 1
            
    t_obj.steps.append(Step("response", final_answer))
    t_obj.db_after = snapshot_db(DB)
    results_mas[task["id"]] = t_obj
    
    print(f"  Прогресс: {idx}/{len(TASKS)} - Задача {task['id']} завершена (Оценка Критика: {critic_feedback.score}/10)")

time_mas = time.time() - t0
cost_mas = n_calls * 0.00025
print(f"\nMAS завершил работу за {time_mas:.1f}с, вызовов инструментов: {n_calls}")

=== Запуск Multi-Agent System (MAS) ===
[GUARD] is_on_topic -> yes
[LOG] user: Get information about booking ABC123
[GUARD] is_on_topic -> yes
  Прогресс: 1/27 - Задача t01 завершена (Оценка Критика: 10.0/10)
[GUARD] is_on_topic -> yes
[LOG] user: search for flights from Moscow to Paris on February 20, 2026
[GUARD] is_on_topic -> yes
  Прогресс: 2/27 - Задача t02 завершена (Оценка Критика: 10.0/10)
[GUARD] is_on_topic -> yes
[LOG] user: Get the status of booking XYZ789
[GUARD] is_on_topic -> yes
  Прогресс: 3/27 - Задача t03 завершена (Оценка Критика: 10.0/10)
[GUARD] is_on_topic -> yes
[LOG] user: Get booking details for booking ABC123
[GUARD] is_on_topic -> yes
[LOG] user: Search for flights from [origin city] to [destination city] on April 16, 2026, departing after the current booking time
[GUARD] is_on_topic -> yes
[LOG] user: Update booking ABC123 to a later flight on April 16, 2026
[GUARD] is_on_topic -> yes
  Прогресс: 4/27 - Задача t04 завершена (Оценка Критика: 10.0/10)
[GUARD

### Оценка MAS (Судья v4)

In [10]:
judge_labels_mas = {}
print("=== Оценка MAS ===")
for task in TASKS:
    tid = task["id"]
    judge_labels_mas[tid] = call_judge_v4(task["query"], results_mas[tid], llm=llm)

avg_score_mas = sum(quality_score(judge_labels_mas[t["id"]]) for t in TASKS) / len(TASKS)
judge_use_mas = [judge_labels_mas[t["id"]]["usefulness"] for t in TASKS]
kappa_use_mas = cohens_kappa(human_use, judge_use_mas)

update_scoreboard("Multi-Agent (MAS)", avg_score_mas, kappa_use_mas, time_mas, n_calls, cost_mas)
print_scoreboard()

=== Оценка MAS ===

                             EVALUATION SCOREBOARD                              
Version                   Avg Score    Cohen Kappa  Time (s)   Calls    Cost ($)
────────────────────────────────────────────────────────────────────────────────
Human GT                  5.07         1.000        0.0s       0        $0.00000
ReAct v1 (Баги)           5.26         0.108        150.7s     48       $0.01200
ReAct v2 (Исправлено)     4.85         0.111        163.9s     57       $0.01425
Multi-Agent (MAS)         5.07         0.180        330.8s     27       $0.00675

